In [ ]:
from google.colab import drive
try:
    drive.mount('/content/drive')   
except Exception as e:
    print('Not running in Colab or mount failed:', e)

In [ ]:
from google.colab import files
uploaded = files.upload() 

In [ ]:
!pip install numpy pandas pyarrow scikit-learn matplotlib seaborn pytest kagglehub

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sobhanmoosavi/us-accidents")

print("Path to dataset files:", path)


In [ ]:
from pathlib import Path
import pandas as pd


dataset_dir = Path(path)

csv_files = list(dataset_dir.rglob("*.csv"))


csv_path = csv_files[0]
df_sample = pd.read_csv(csv_path, nrows=10_000)

print(df_sample.shape)
df_sample.head()
df_sample.info()
df_sample.describe()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Work on a copy
corr_df = df_sample.copy()

# --------------------------------------------------
# 1. Parse Start_Time and engineer useful time fields
# --------------------------------------------------
corr_df["Start_Time"] = pd.to_datetime(
    corr_df["Start_Time"],
    errors="coerce"
)

corr_df["hour"] = corr_df["Start_Time"].dt.hour
corr_df["month"] = corr_df["Start_Time"].dt.month
corr_df["day_of_week"] = corr_df["Start_Time"].dt.dayofweek

corr_df["is_weekend"] = (
    corr_df["day_of_week"].isin([5, 6]).astype("int8")
)

# Cyclical time features
corr_df["hour_sin"] = np.sin(
    2 * np.pi * corr_df["hour"] / 24
)
corr_df["hour_cos"] = np.cos(
    2 * np.pi * corr_df["hour"] / 24
)

corr_df["month_sin"] = np.sin(
    2 * np.pi * corr_df["month"] / 12
)
corr_df["month_cos"] = np.cos(
    2 * np.pi * corr_df["month"] / 12
)

# --------------------------------------------------
# 2. Convert day/night to binary
# --------------------------------------------------
corr_df["is_day"] = corr_df["Sunrise_Sunset"].map({
    "Day": 1,
    "Night": 0
})

# --------------------------------------------------
# 3. Convert boolean road features to 0/1
# --------------------------------------------------
boolean_columns = [
    "Amenity",
    "Bump",
    "Crossing",
    "Give_Way",
    "Junction",
    "No_Exit",
    "Railway",
    "Roundabout",
    "Station",
    "Stop",
    "Traffic_Calming",
    "Traffic_Signal",
    "Turning_Loop",
]

for column in boolean_columns:
    corr_df[column] = corr_df[column].astype("int8")

# --------------------------------------------------
# 4. Select correlation columns
# --------------------------------------------------
correlation_columns = [
    "Severity",
    "Start_Lat",
    "Start_Lng",
    "Distance(mi)",
    "Temperature(F)",
    "Humidity(%)",
    "Pressure(in)",
    "Visibility(mi)",
    "Wind_Speed(mph)",
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
    "is_weekend",
    "is_day",
] + boolean_columns

# Keep only columns that exist
correlation_columns = [
    column
    for column in correlation_columns
    if column in corr_df.columns
]

analysis_df = corr_df[correlation_columns].copy()

# --------------------------------------------------
# 5. Remove columns with no variation
# --------------------------------------------------
constant_columns = [
    column
    for column in analysis_df.columns
    if analysis_df[column].nunique(dropna=True) <= 1
]

if constant_columns:
    print("Removed constant columns:")
    print(constant_columns)

analysis_df = analysis_df.drop(columns=constant_columns)

# --------------------------------------------------
# 6. Calculate Spearman correlations
# --------------------------------------------------
correlation_matrix = analysis_df.corr(
    method="spearman",
    min_periods=100
)

# Display correlations with Severity
severity_correlations = (
    correlation_matrix["Severity"]
    .drop("Severity")
    .sort_values(key=lambda values: values.abs(), ascending=False)
)

print("\nCorrelations with Severity:")
print(severity_correlations.round(3))

# --------------------------------------------------
# 7. Plot the full correlation matrix
# --------------------------------------------------
plt.figure(figsize=(18, 14))

sns.heatmap(
    correlation_matrix,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    annot=False,
    square=False,
    linewidths=0.25,
)

plt.title("Spearman Correlation Matrix")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 8. Plot correlations with Severity only
# --------------------------------------------------
plt.figure(figsize=(9, 8))
 
severity_correlations.sort_values().plot(
    kind="barh",
    color=[
        "steelblue" if value >= 0 else "indianred"
        for value in severity_correlations.sort_values()
    ],
)

plt.axvline(0, color="black", linewidth=1)
plt.xlabel("Spearman Correlation")
plt.ylabel("Feature")
plt.title("Feature Correlations with Severity")
plt.tight_layout()
plt.show()



In [ ]:
df_sample["visibility_group"] = pd.cut(
    df_sample["Visibility(mi)"],
    bins=[0, 1, 3, 5, 10, float("inf")],
    labels=["<1", "1-3", "3-5", "5-10", "10+"]
)

pd.crosstab(
    df_sample["visibility_group"],
    df_sample["Severity"],
    normalize="index"
)

In [ ]:
df = df.loc[df['city'] == 'Atlanta']
